In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'], check=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

TICKER = 'LKOH.ME'
START_DATE = '2010-01-01'
CSV_FILE = '1luk.csv'

win_len = 30
batch_size = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def download_close_series(ticker, start_date, file_name):
    quotes = yf.download(ticker, start=start_date)

    if isinstance(quotes.columns, pd.MultiIndex):
        close_series = quotes[('Close', ticker)]
    else:
        close_series = quotes['Close']

    table = close_series.to_frame(name='CLOSE')
    table.to_csv(file_name)

    print(f'Данные сохранены в {file_name}')
    return table

def read_close_values(file_name):
    frame = pd.read_csv(file_name)
    return frame['CLOSE'].values.reshape(-1, 1)

def normalize_values(values):
    scaler_obj = StandardScaler()
    transformed = scaler_obj.fit_transform(values)
    return transformed, scaler_obj

def make_one_step_dataset(values, window, border):
    x_items = []
    y_items = []

    for pos in range(len(values) - window):
        x_items.append(values[pos:pos + window])
        y_items.append(values[pos + window])

    x_items = np.array(x_items, dtype=np.float32)
    y_items = np.array(y_items, dtype=np.float32)

    train_stop = border - window + 1
    test_start = border

    return x_items[:train_stop], y_items[:train_stop], x_items[test_start:], y_items[test_start:]

def make_loader(x, y, batch, shuffle=True):
    x_tensor = torch.tensor(x, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch, shuffle=shuffle)

def train_model(model, loader, epochs, learning_rate=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_count = 0

        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            prediction = model(x_batch)
            loss = criterion(prediction, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(x_batch)
            total_count += len(x_batch)

        epoch_loss = total_loss / total_count
        losses.append(epoch_loss)
        print(f'Эпоха {epoch + 1}/{epochs}, loss = {epoch_loss:.6f}')

    return losses

def predict_model(model, x, batch=256):
    model.eval()
    predictions = []
    loader = DataLoader(torch.tensor(x, dtype=torch.float32), batch_size=batch, shuffle=False)

    with torch.no_grad():
        for x_batch in loader:
            x_batch = x_batch.to(device)
            output = model(x_batch).cpu().numpy()
            predictions.append(output)

    return np.vstack(predictions)

def plot_loss(values, title):
    plt.figure(figsize=(8, 4))
    plt.plot(values)
    plt.title(title)
    plt.xlabel('Эпоха')
    plt.ylabel('MSE')
    plt.show()

download_close_series(TICKER, START_DATE, CSV_FILE)

data = read_close_values(CSV_FILE)
data_scaled, scaler = normalize_values(data)
train_len = int(len(data_scaled) * 0.8)

x_train_1, y_train_1, x_test_1, y_test_1 = make_one_step_dataset(data_scaled, win_len, train_len)
train_loader_1 = make_loader(x_train_1, y_train_1, batch_size)


In [ ]:
class DenseOneStep(nn.Module):
    def __init__(self, window):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(window * 64, 1)
        )

    def forward(self, x):
        return self.net(x)

def show_single_step_forecast(real_values, predicted_values):
    plt.figure(figsize=(10, 5))
    plt.plot(real_values, label='Оригинал')
    plt.plot(predicted_values, label='Предсказание Dense')
    plt.title('Прогноз на 1 шаг (Dense, PyTorch)')
    plt.legend()
    plt.show()

model_dense = DenseOneStep(win_len)
dense_history = train_model(model_dense, train_loader_1, epochs=10)
plot_loss(dense_history, 'Ошибка Dense-модели на 1 шаг')

dense_predictions = predict_model(model_dense, x_test_1)

y_true = data[train_len + win_len:]
y_pred = scaler.inverse_transform(dense_predictions)

show_single_step_forecast(y_true, y_pred)


In [ ]:
future_steps = 10

def make_multi_step_dataset(values, window, horizon):
    x_items = []
    y_items = []
    last_pos = len(values) - window - horizon

    for pos in range(last_pos):
        x_items.append(values[pos:pos + window])
        y_items.append(values[pos + window:pos + window + horizon])

    x_items = np.array(x_items, dtype=np.float32)
    y_items = np.array(y_items, dtype=np.float32).reshape(-1, horizon)

    return x_items, y_items

class DenseTenSteps(nn.Module):
    def __init__(self, window, horizon):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(window * 128, horizon)
        )

    def forward(self, x):
        return self.net(x)

def show_multi_step_forecast(real_values, predicted_values, horizon):
    fig, axes = plt.subplots(5, 2, figsize=(15, 20))
    axes = axes.flatten()

    for step in range(horizon):
        axes[step].plot(real_values[:100, step], label='Факт')
        axes[step].plot(predicted_values[:100, step], label='Прогноз')
        axes[step].set_title(f'Шаг предсказания: {step + 1}')
        axes[step].legend()

    plt.tight_layout()
    plt.show()

x_all_10, y_all_10 = make_multi_step_dataset(data_scaled, win_len, future_steps)

x_train_10 = x_all_10[:train_len]
y_train_10 = y_all_10[:train_len]
x_test_10 = x_all_10[train_len:]
y_test_10 = y_all_10[train_len:]

train_loader_10 = make_loader(x_train_10, y_train_10, batch_size)

model_10 = DenseTenSteps(win_len, future_steps)
dense_10_history = train_model(model_10, train_loader_10, epochs=15)
plot_loss(dense_10_history, 'Ошибка Dense-модели на 10 шагов')

pred_10 = predict_model(model_10, x_test_10)

pred_10_rescaled = scaler.inverse_transform(pred_10)
true_10_rescaled = scaler.inverse_transform(y_test_10)

show_multi_step_forecast(true_10_rescaled, pred_10_rescaled, future_steps)


In [ ]:
class LstmTenSteps(nn.Module):
    def __init__(self, horizon):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, horizon)
        )

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        x = hidden[-1]
        x = self.dropout(x)
        return self.head(x)

def show_lstm_result(real_values, predicted_values):
    plt.figure(figsize=(10, 5))
    plt.plot(real_values[:100, 0], label='Оригинал (шаг 1)')
    plt.plot(predicted_values[:100, 0], label='LSTM прогноз (шаг 1)')
    plt.title('Результат работы LSTM на PyTorch')
    plt.legend()
    plt.show()

model_lstm = LstmTenSteps(future_steps)
lstm_history = train_model(model_lstm, train_loader_10, epochs=15)
plot_loss(lstm_history, 'Ошибка LSTM-модели')

pred_lstm = predict_model(model_lstm, x_test_10)
pred_lstm_rescaled = scaler.inverse_transform(pred_lstm)

show_lstm_result(true_10_rescaled, pred_lstm_rescaled)
